## Set up AnnData object with integration winner, and plot UMAP

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_py_analysis
# python -m ipykernel install --user --name scrna_cartography_py_analysis --display-name "py_analysis"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad         # Core data structure for single-cell data
import scanpy as sc          # Analysis and visualization of single-cell data

# Data visualization
import seaborn as sns        # Statistical data visualization
import matplotlib.pyplot as plt  # Plotting interface
import matplotlib            # Base matplotlib functionality
from matplotlib.backends.backend_pdf import PdfPages  # Save plots to multi-page PDFs
import tol_colors            # colors

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# Dataframes
import pandas as pd

# Miscellaneous utilities
import warnings              # Suppress or manage warnings

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import my_anndata as ma                    # Custom AnnData utilities

Parameters

In [2]:
base_dir = str(here('data/integrate/third_pass/'))
plot_dir = os.path.join(base_dir, 'plot') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

In [3]:
# Plot style
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# Font sizes matching ggplot theme
SMALL_SIZE = 6     # axis text ~6 pt
MEDIUM_SIZE = 7    # axis/strip title ~7 pt
BIGGER_SIZE = 8    # plot title ~8 pt

plt.rc('font', size=SMALL_SIZE)           # default text
plt.rc('axes', titlesize=BIGGER_SIZE)     # axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)     # axis labels
plt.rc('xtick', labelsize=SMALL_SIZE)     # x tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)     # y tick labels
plt.rc('legend', fontsize=SMALL_SIZE)     # legend text
plt.rc('figure', titlesize=BIGGER_SIZE)   # figure title

# Scanpy figure parameters
sc.set_figure_params(figsize=(3, 3), frameon=False, dpi_save=300)
sc.settings.figdir = plot_dir

# Inline backend
%config InlineBackend.print_figure_kwargs = {'facecolor': 'w'}
%config InlineBackend.figure_format = 'retina'

In [4]:
embedding_keys = ['X_latent_1']

Import data

In [ ]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AF_combined.h5ad"))

In [6]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AG_combined.h5ad"))

In [7]:
module_score = pd.read_csv(here('data/quality_control_per_cluster/first_pass/files/marker_gene_scores.csv'), index_col= 0)
module_score = module_score[[column for column in module_score.columns if column.startswith('azimuth')]]
doublet_probs = pd.read_csv(here('data/quality_control_per_cluster/first_pass/files/doublet_probabilities.csv'), index_col= 'barcode')

In [8]:
adata.obs = adata.obs.join(
    doublet_probs[["doublet_probability"]],
    how="left")

In [9]:
adata.obs = adata.obs.join(module_score)

Find neighbors

In [ ]:
# Function to compute neighbors for one embedding
def compute_neighbors(key):
    neig_key = f"{key}_neighbors"
    sc.pp.neighbors(
        adata, 
        use_rep=key, 
        key_added=neig_key
    )

# Run in parallel with threading backend
with parallel_backend("threading", n_jobs=60):  # controls number of threads
    Parallel(n_jobs=len(embedding_keys))(
        delayed(compute_neighbors)(k) for k in embedding_keys
    )

Compute UMAP

In [ ]:
# Function to compute UMAP for one neighbors graph
def compute_umap(neighbors_key):
    umap_key = neighbors_key.replace("_neighbors", "_umap")
    sc.tl.umap(
        adata, 
        neighbors_key=neighbors_key, 
        key_added=umap_key
    )

neighbors_keys = [f"{key}_neighbors" for key in embedding_keys]

# Run sequentially 
for nk in neighbors_keys:
    compute_umap(nk)

Save adata object with neighbors and umap

In [ ]:
adata.write(os.path.join(anndata_dir, "AG_combined.h5ad"))

In [10]:
from matplotlib.backends.backend_pdf import PdfPages
import re

with PdfPages(os.path.join(plot_dir, 'latent_1_umap.pdf')) as pdf:
    
    # study_cell_annotation_harmonized
    fig, colors = ma.umap_single(
        ad=adata,
        variable='study_cell_annotation_harmonized',
        pt_size=1,
        exclude=True,
        exclude_key="unknown",
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')

    fig, colors = ma.umap_facet(
        ad=adata,
        variable='study_cell_annotation_harmonized',
        ncols=6,
        pt_size=5,
        exclude=True,
        exclude_key="unknown",
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # cell_nuclei
    fig, colors = ma.umap_single(
        ad=adata,
        variable='cell_nuclei',
        pt_size=1,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='cell_nuclei',
        ncols=2,
        pt_size=5,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # ic_id_study
    fig, colors = ma.umap_single(
        ad=adata,
        variable='ic_id_study',
        pt_size=1,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='ic_id_study',
        ncols=6,
        pt_size=5,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # library_prep
    fig, colors = ma.umap_single(
        ad=adata,
        variable='library_prep',
        pt_size=1,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='library_prep',
        ncols=6,
        pt_size=5,
        umap_key='X_latent_1_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # Module scores
    mod_scores = [col for col in module_score.columns if col.startswith('azimuth')]
    sc.pl.embedding(
        adata,
        basis="X_latent_1_umap",
        title=[re.sub('azimuth_', '', val) for val in mod_scores],
        color=mod_scores,
        cmap=tol_colors.iridescent,
        size=1,
        frameon=False,
        show=False
    )
    plt.tight_layout()
    pdf.savefig(plt.gcf(), bbox_inches='tight')
    
    plt.close('all')


/tmp/ipykernel_256415/2107884192.py:118: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [11]:
with PdfPages(os.path.join(plot_dir, 'latent_10_umap.pdf')) as pdf:
    
    # study_cell_annotation_harmonized
    fig, colors = ma.umap_single(
        ad=adata,
        variable='study_cell_annotation_harmonized',
        pt_size=1,
        exclude=True,
        exclude_key="unknown",
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')

    fig, colors = ma.umap_facet(
        ad=adata,
        variable='study_cell_annotation_harmonized',
        ncols=6,
        pt_size=5,
        exclude=True,
        exclude_key="unknown",
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # cell_nuclei
    fig, colors = ma.umap_single(
        ad=adata,
        variable='cell_nuclei',
        pt_size=1,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='cell_nuclei',
        ncols=2,
        pt_size=5,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # ic_id_study
    fig, colors = ma.umap_single(
        ad=adata,
        variable='ic_id_study',
        pt_size=1,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='ic_id_study',
        ncols=6,
        pt_size=5,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # library_prep
    fig, colors = ma.umap_single(
        ad=adata,
        variable='library_prep',
        pt_size=1,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    fig, colors = ma.umap_facet(
        ad=adata,
        variable='library_prep',
        ncols=6,
        pt_size=5,
        umap_key='X_latent_10_umap'
    )
    ax = fig
    fig_obj = ax.figure if hasattr(ax, "figure") else ax
    fig_obj.tight_layout()
    pdf.savefig(fig_obj, bbox_inches='tight')
    
    # Module scores
    mod_scores = [col for col in module_score.columns if col.startswith('azimuth')]
    sc.pl.embedding(
        adata,
        basis="X_latent_10_umap",
        title=[re.sub('azimuth_', '', val) for val in mod_scores],
        color=mod_scores,
        cmap=tol_colors.iridescent,
        size=1,
        frameon=False,
        show=False
    )
    plt.tight_layout()
    pdf.savefig(plt.gcf(), bbox_inches='tight')
    
    plt.close('all')


/tmp/ipykernel_256415/1416552521.py:115: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [ ]:
# test = adata.obs[["ic_id_study", "name"]].copy().reset_index()
# test[["ic_id_study", "name"]].drop_duplicates()

adata.obs[["ic_id_study"]].value_counts()